# Fused Flash Attention Forward Pass

**Time Limit: 60 minutes** | **GPU: NVIDIA T4 (SM75)**

---

## Background

In this exercise, you will implement core components of the **FlashAttention-2** algorithm — a fused, tiled, memory-efficient attention kernel.

Standard attention computes:
```
O = softmax(Q @ K^T / sqrt(d)) @ V
```

The naive approach materializes the full `N×N` attention matrix, which is prohibitive for long sequences. FlashAttention avoids this by:

1. Tiling the K/V matrices into blocks and processing them sequentially
2. Using the online softmax algorithm to compute numerically stable softmax incrementally, without ever forming the full attention matrix
3. Fusing both GEMMs and the softmax into a single kernel

### The Online Softmax Algorithm

For each query row, we maintain running statistics:
- `m_i` — the running max of `Q·K^T` scores seen so far
- `l_i` — the running sum of `exp(scores - m_i)` seen so far

When processing a new block of K with scores `s_new`:
1. Compute `m_new = max(m_i, max(s_new))`
2. Rescale: `l_new = l_i * exp(m_i - m_new) + sum(exp(s_new - m_new))`
3. Rescale previous output: `O = O * (l_i * exp(m_i - m_new) / l_new)` 
4. Accumulate: `O += exp(s_new - m_new) / l_new * V_block`

In practice, steps 3–4 are split: we defer the `1/l_new` division to the very end.

---

## Candidate Instructions

We provide:
- A naive reference kernel that materializes the full attention matrix
- A tiled kernel skeleton with `TODO` sections for you to fill in

You must complete 3 tasks in the tiled kernel:

| Task | Description | Points |
|------|-------------|--------|
| **A** | Load Q, K, V tiles into shared memory with correct indexing | 25 |
| **B** | Implement the online softmax: running max, rescaling of O, and exp-sum | 50 |
| **C** | Final normalization and GMEM write-out | 25 |

Your kernel should produce results matching the naive reference within `atol=1e-2, rtol=1e-1` (FP16 tolerance).

---

## Evaluation Criteria

1. Correctness (60%) — passes the numerical verification against reference
2. Code Quality (20%) — clean, well-reasoned implementation; correct use of shared memory and synchronization
3. Performance Awareness (20%) — understanding of bank conflicts, coalescing, occupancy tradeoffs

## Setup

Run the cells below to verify your GPU and compile the starter code.

In [ ]:
!nvidia-smi --query-gpu=name,compute_cap --format=csv,noheader

In [ ]:
!nvcc --version

## Part 0: Naive Reference Implementation

Study this carefully — it shows the exact computation your tiled kernel must replicate.

In [ ]:
%%writefile fmha_interview.cu
#include <cuda_fp16.h>
#include <cuda_runtime.h>
#include <math.h>
#include <stdio.h>
#include <stdlib.h>
#include <float.h>

#define CHECK_CUDA(call)                                                       \
  do {                                                                         \
    cudaError_t err = call;                                                    \
    if (err != cudaSuccess) {                                                  \
      fprintf(stderr, "CUDA error at %s:%d: %s\n", __FILE__, __LINE__,      \
              cudaGetErrorString(err));                                         \
      exit(1);                                                                 \
    }                                                                          \
  } while (0)

// ============================================================================
// CONSTANTS
// ============================================================================
// Problem dimensions
constexpr int BATCH = 2;
constexpr int HEADS = 4;
constexpr int SEQ_LEN = 256;   // N = sequence length
constexpr int HEAD_DIM = 64;   // d = head dimension

// Tiling parameters for the candidate's kernel
constexpr int BLK_M = 32;  // tile rows of Q
constexpr int BLK_N = 32;  // tile cols of K (number of keys per tile)
constexpr int BLK_D = 64;  // == HEAD_DIM (full head dim loaded)

// ============================================================================
// UTILITY: Convert float <-> half
// ============================================================================
__device__ __forceinline__ float h2f(half x) { return __half2float(x); }
__device__ __forceinline__ half f2h(float x) { return __float2half(x); }

// ============================================================================
// NAIVE REFERENCE KERNEL (DO NOT MODIFY)
// Computes attention for one (batch, head) per thread block.
// Materializes the full S = Q @ K^T matrix in registers (only works for small N).
// ============================================================================
__global__ void fmha_naive_kernel(
    const half* __restrict__ Q,   // [B, H, N, d]
    const half* __restrict__ K,   // [B, H, N, d]
    const half* __restrict__ V,   // [B, H, N, d]
    half* __restrict__ O,         // [B, H, N, d]
    int N, int d, float scale)
{
    // Each block handles one (batch, head) pair.
    // Each thread handles one query row.
    int bh = blockIdx.x;  // combined batch*head index
    int b = bh / HEADS;
    int h = bh % HEADS;
    int row = threadIdx.x;  // query row this thread handles
    if (row >= N) return;

    int bhOffset = (b * HEADS + h) * N * d;

    // Step 1: Compute S[row, :] = Q[row, :] @ K^T = sum_k Q[row,k]*K[col,k]
    float scores[SEQ_LEN];  // N is small enough for registers
    float max_val = -FLT_MAX;
    for (int col = 0; col < N; col++) {
        float sum = 0.0f;
        for (int k = 0; k < d; k++) {
            sum += h2f(Q[bhOffset + row * d + k]) *
                   h2f(K[bhOffset + col * d + k]);
        }
        scores[col] = sum * scale;
        max_val = fmaxf(max_val, scores[col]);
    }

    // Step 2: Softmax
    float sum_exp = 0.0f;
    for (int col = 0; col < N; col++) {
        scores[col] = expf(scores[col] - max_val);
        sum_exp += scores[col];
    }
    for (int col = 0; col < N; col++) {
        scores[col] /= sum_exp;
    }

    // Step 3: O[row, :] = P[row, :] @ V
    for (int k = 0; k < d; k++) {
        float sum = 0.0f;
        for (int col = 0; col < N; col++) {
            sum += scores[col] * h2f(V[bhOffset + col * d + k]);
        }
        O[bhOffset + row * d + k] = f2h(sum);
    }
}

// ============================================================================
// CANDIDATE KERNEL: Tiled Flash Attention with Online Softmax
// ============================================================================
//
// Grid:  (B * H) blocks
// Block: (BLK_M) threads, one thread per query row in the tile.
//        We loop over query tiles in the grid-stride pattern.
//
// Shared memory layout:
//   sQ[BLK_M][BLK_D]  — current Q tile
//   sK[BLK_N][BLK_D]  — current K tile
//   sV[BLK_N][BLK_D]  — current V tile
//
// Algorithm:
//   For each Q-tile (rows [qStart, qStart+BLK_M)):
//     Initialize m_i = -inf, l_i = 0, O_i = 0
//     For each K/V-tile (cols [kvStart, kvStart+BLK_N)):
//       1. Load sK, sV from global memory
//       2. Compute s = sQ @ sK^T (BLK_M x BLK_N scores)
//       3. Online softmax update: m_i, l_i, rescale O_i
//       4. Accumulate: O_i += diag(exp(s - m_new)) @ sV
//     Finalize: O_i /= l_i
//     Write O_i to global memory
//
// NOTE: This uses a simple thread-per-row model to keep
// the focus on the online softmax algorithm and tiling logic. A production
// kernel would use MMA instructions
// ============================================================================

__global__ void fmha_tiled_kernel(
    const half* __restrict__ Q,
    const half* __restrict__ K,
    const half* __restrict__ V,
    half* __restrict__ O,
    int N, int d, float scale)
{
    // Shared memory for tiles
    __shared__ float sQ[BLK_M][BLK_D];
    __shared__ float sK[BLK_N][BLK_D];
    __shared__ float sV[BLK_N][BLK_D];

    int bh = blockIdx.x;  // combined (batch, head) index
    int b = bh / HEADS;
    int h = bh % HEADS;
    int tid = threadIdx.x;

    int bhOffset = (b * HEADS + h) * N * d;

    // Number of K/V tiles
    int numKVTiles = (N + BLK_N - 1) / BLK_N;

    // Loop over Q-tiles (each block processes all Q-tiles for its batch/head)
    for (int qTile = 0; qTile < (N + BLK_M - 1) / BLK_M; qTile++) {
        int qStart = qTile * BLK_M;
        int qRow = qStart + tid;  // global row index for this thread

        // ================================================================
        // TASK A: Load Q tile into shared memory (10 lines)
        // ================================================================
        // Load sQ[tid][0..d-1] from Q[bhOffset + qRow * d + k]
        // Handle the case where qRow >= N (pad with zeros)
        //
        // HINT: Each thread loads its own row of Q into sQ[tid][:].
        //       Use h2f() to convert half -> float.
        //
        // >>> TODO A: WRITE YOUR CODE HERE <<<
        // ---- START TASK A ----

        // ---- END TASK A ----

        __syncthreads();

        // Per-thread accumulators for online softmax.
        float row_max = -FLT_MAX;     // m_i: running max
        float row_sum = 0.0f;         // l_i: running sum of exp
        float acc[BLK_D];             // O_i: running weighted sum
        for (int i = 0; i < BLK_D; i++) acc[i] = 0.0f;

        // ================================================================
        // Loop over K/V tiles
        // ================================================================
        for (int kvTile = 0; kvTile < numKVTiles; kvTile++) {
            int kvStart = kvTile * BLK_N;

            // ------------------------------------------------------------
            // TASK A (continued): Load K and V tiles into shared memory
            // ------------------------------------------------------------
            // Load sK[tid][0..d-1] and sV[tid][0..d-1].
            // Since BLK_N == BLK_M == 32 and we have 32 threads,
            // each thread loads one row of K and one row of V.
            // Handle kvStart + tid >= N by padding with zeros.
            //
            // >>> TODO A (continued): WRITE YOUR CODE HERE <<<
            // ---- START TASK A (continued) ----

            // ---- END TASK A (continued) ----

            __syncthreads();

            // Compute scores: s[j] = sum_k sQ[tid][k] * sK[j][k]
            // This is one row of (Q_tile @ K_tile^T)
            float scores[BLK_N];
            for (int j = 0; j < BLK_N; j++) {
                float dot = 0.0f;
                for (int k = 0; k < d; k++) {
                    dot += sQ[tid][k] * sK[j][k];
                }
                scores[j] = dot * scale;
                // Mask out-of-bounds keys
                if (kvStart + j >= N) scores[j] = -FLT_MAX;
            }

            // ============================================================
            // TASK B: Online Softmax Update (~25 lines)
            // ============================================================
            //
            // Given the current tile's `scores[0..BLK_N-1]`, update:
            //   row_max (m_i), row_sum (l_i), acc[0..d-1] (O_i)
            //
            // Steps:
            //   1. Find m_new = max(row_max, max(scores[:]))
            //
            //   2. Compute the rescaling factor for the OLD accumulator:
            //        alpha = exp(row_max - m_new)
            //      This corrects the previously accumulated exp values
            //      from the old max to the new max.
            //
            //   3. Rescale the running sum:
            //        row_sum = row_sum * alpha
            //
            //   4. Rescale the output accumulator:
            //        acc[k] *= alpha   for all k in [0, d)
            //
            //   5. For each key j in [0, BLK_N):
            //        p_j = exp(scores[j] - m_new)
            //        row_sum += p_j
            //        acc[k] += p_j * sV[j][k]   for all k in [0, d)
            //
            //   6. Update: row_max = m_new
            //
            // This is the core of FlashAttention. Think carefully about
            // why this produces the same result as the standard softmax.
            //
            // >>> TODO B: WRITE YOUR CODE HERE (replace the next line) <<<
            // ---- START TASK B ----

            // ---- END TASK B ----

            __syncthreads();  // Ensure shared memory is safe to overwrite
        }  // end kvTile loop

        // ================================================================
        // TASK C: Final normalization and write to global memory (~8 lines)
        // ================================================================
        //
        // At this point:
        //   acc[k] = sum over all KV tiles of exp(s_j - row_max) * V[j][k]
        //   row_sum = sum over all KV tiles of exp(s_j - row_max)
        //
        // To get the final softmax-weighted output, divide:
        //   O[qRow][k] = acc[k] / row_sum
        //
        // Write to O[bhOffset + qRow * d + k], using f2h() to convert.
        // Guard against qRow >= N.
        //
        // >>> TODO C: WRITE YOUR CODE HERE <<<
        // ---- START TASK C ----

        // ---- END TASK C ----

        __syncthreads();  // Before loading next Q tile
    }  // end qTile loop
}


// ============================================================================
// HOST CODE: Initialization, Launch, Verification
// ============================================================================

void fill_random_half(half* d_ptr, int n, unsigned seed) {
    float* h_buf = (float*)malloc(n * sizeof(float));
    srand(seed);
    for (int i = 0; i < n; i++) {
        h_buf[i] = (float)(rand() % 1000 - 500) / 1000.0f;  // [-0.5, 0.5]
    }
    half* h_half = (half*)malloc(n * sizeof(half));
    for (int i = 0; i < n; i++) h_half[i] = __float2half(h_buf[i]);
    CHECK_CUDA(cudaMemcpy(d_ptr, h_half, n * sizeof(half), cudaMemcpyHostToDevice));
    free(h_buf);
    free(h_half);
}

bool verify(half* d_ref, half* d_test, int n, float atol, float rtol) {
    half* h_ref = (half*)malloc(n * sizeof(half));
    half* h_test = (half*)malloc(n * sizeof(half));
    CHECK_CUDA(cudaMemcpy(h_ref, d_ref, n * sizeof(half), cudaMemcpyDeviceToHost));
    CHECK_CUDA(cudaMemcpy(h_test, d_test, n * sizeof(half), cudaMemcpyDeviceToHost));

    int errors = 0;
    int first_error = -1;
    for (int i = 0; i < n; i++) {
        float ref_val = __half2float(h_ref[i]);
        float test_val = __half2float(h_test[i]);
        float diff = fabsf(ref_val - test_val);
        float ref_abs = fabsf(ref_val) + 1e-6f;
        if (diff > atol && diff / ref_abs > rtol) {
            errors++;
            if (first_error < 0) first_error = i;
        }
    }

    if (errors > 0) {
        float ref_val = __half2float(h_ref[first_error]);
        float test_val = __half2float(h_test[first_error]);
        printf("FAILED: %d/%d errors (%.2f%%). First error at [%d]: ref=%.6f, got=%.6f\n",
               errors, n, 100.0f * errors / n, first_error, ref_val, test_val);
    } else {
        printf("PASSED: All %d values match (atol=%.0e, rtol=%.0e)\n", n, atol, rtol);
    }

    free(h_ref);
    free(h_test);
    return errors == 0;
}

int main() {
    int N = SEQ_LEN;
    int d = HEAD_DIM;
    int total = BATCH * HEADS * N * d;
    float scale = 1.0f / sqrtf((float)d);

    printf("=== Flash Attention Interview Test ===\n");
    printf("B=%d, H=%d, N=%d, d=%d, scale=%.4f\n", BATCH, HEADS, N, d, scale);
    printf("Tile: BLK_M=%d, BLK_N=%d, BLK_D=%d\n\n", BLK_M, BLK_N, BLK_D);

    half *d_Q, *d_K, *d_V, *d_O_ref, *d_O_test;
    CHECK_CUDA(cudaMalloc(&d_Q, total * sizeof(half)));
    CHECK_CUDA(cudaMalloc(&d_K, total * sizeof(half)));
    CHECK_CUDA(cudaMalloc(&d_V, total * sizeof(half)));
    CHECK_CUDA(cudaMalloc(&d_O_ref, total * sizeof(half)));
    CHECK_CUDA(cudaMalloc(&d_O_test, total * sizeof(half)));

    fill_random_half(d_Q, total, 42);
    fill_random_half(d_K, total, 123);
    fill_random_half(d_V, total, 456);
    CHECK_CUDA(cudaMemset(d_O_ref, 0, total * sizeof(half)));
    CHECK_CUDA(cudaMemset(d_O_test, 0, total * sizeof(half)));

    // --- Run naive reference ---
    printf("Running naive reference kernel...\n");
    fmha_naive_kernel<<<BATCH * HEADS, N>>>(d_Q, d_K, d_V, d_O_ref, N, d, scale);
    CHECK_CUDA(cudaDeviceSynchronize());
    printf("Naive kernel done.\n\n");

    // --- Run candidate's tiled kernel ---
    printf("Running tiled kernel (candidate solution)...\n");
    fmha_tiled_kernel<<<BATCH * HEADS, BLK_M>>>(d_Q, d_K, d_V, d_O_test, N, d, scale);
    CHECK_CUDA(cudaDeviceSynchronize());
    printf("Tiled kernel done.\n\n");

    // --- Verify ---
    printf("Verification:\n");
    bool ok = verify(d_O_ref, d_O_test, total, 1e-2f, 1e-1f);

    // --- Timing ---
    if (ok) {
        printf("\nTiming (100 iterations):\n");
        cudaEvent_t start, stop;
        cudaEventCreate(&start);
        cudaEventCreate(&stop);

        // Warm up
        for (int i = 0; i < 10; i++)
            fmha_tiled_kernel<<<BATCH * HEADS, BLK_M>>>(d_Q, d_K, d_V, d_O_test, N, d, scale);
        cudaDeviceSynchronize();

        cudaEventRecord(start);
        for (int i = 0; i < 100; i++)
            fmha_tiled_kernel<<<BATCH * HEADS, BLK_M>>>(d_Q, d_K, d_V, d_O_test, N, d, scale);
        cudaEventRecord(stop);
        cudaEventSynchronize(stop);
        float ms;
        cudaEventElapsedTime(&ms, start, stop);
        printf("  Tiled kernel: %.4f ms avg\n", ms / 100.0f);

        // Naive timing for comparison
        for (int i = 0; i < 10; i++)
            fmha_naive_kernel<<<BATCH * HEADS, N>>>(d_Q, d_K, d_V, d_O_ref, N, d, scale);
        cudaDeviceSynchronize();

        cudaEventRecord(start);
        for (int i = 0; i < 100; i++)
            fmha_naive_kernel<<<BATCH * HEADS, N>>>(d_Q, d_K, d_V, d_O_ref, N, d, scale);
        cudaEventRecord(stop);
        cudaEventSynchronize(stop);
        cudaEventElapsedTime(&ms, start, stop);
        printf("  Naive kernel: %.4f ms avg\n", ms / 100.0f);

        cudaEventDestroy(start);
        cudaEventDestroy(stop);
    }

    CHECK_CUDA(cudaFree(d_Q));
    CHECK_CUDA(cudaFree(d_K));
    CHECK_CUDA(cudaFree(d_V));
    CHECK_CUDA(cudaFree(d_O_ref));
    CHECK_CUDA(cudaFree(d_O_test));

    return ok ? 0 : 1;
}


## Compile & Run

Edit the TODO sections in the cell above, then run the cells below to compile and test.

In [ ]:
!nvcc -O3 -arch=sm_75 -o fmha_interview fmha_interview.cu && echo "Compilation successful!"

In [ ]:
!./fmha_interview